# 🧠 Aula 13 — Implementação de um Modelo Paralelo Simples (Síntese do Bloco 2)

**Curso:** Tecnologia e Infraestrutura para Inteligência Artificial — Senac  
**Bloco 2:** Programação, Otimização e Computação Heterogênea  

---

## 🎯 Objetivo da Aula
Desenvolver e comparar implementações paralelas (**GPU — CUDA Numba e CuPy**) e sequenciais (**CPU — Python puro e NumPy**) da **soma vetorial** e do **produto escalar**, gerando gráficos de speedup, um mini-relatório técnico e resolvendo **20 exercícios práticos e teóricos** para consolidar os aprendizados do Bloco 2.

### ⚙️ PASSO CRÍTICO: Ativar a GPU no Google Colab
1. Clique no menu superior **Ambiente de execução** (*Runtime*) ➔ **Alterar tipo de ambiente de execução** (*Change runtime type*).
2. Em **Acelerador de hardware**, escolha **T4 GPU** (ou equivalente NVIDIA).
3. Clique em **Salvar**.


## 1. Contextualização Teórica: Por que a GPU ganha? (SIMD vs. SIMT)

- **CPU — SIMD (Single Instruction, Multiple Data) / MIMD:** 8–16 núcleos potentes com execução sequencial por padrão ou vetorial curta (AVX-512).
- **GPU — SIMT (Single Instruction, Multiple Threads):** Milhares de threads em paralelo organizadas em Warps (32 threads) executando a mesma instrução sobre uma grade multidimensional.

Na soma vetorial $c[i] = a[i] + b[i]$ para $N = 10.000.000$:
- **CPU:** 10M operações efetuadas em série ou em pequenos blocos de núcleos.
- **GPU:** 10M operações distribuídas em dezenas de milhares de threads simultâneas em batches massivos.


## 2. Versão 1 & 2 — CPU: Python Puro e NumPy

Abaixo definimos as implementações de referência em CPU:
1. **Python Puro:** Laços nativos `for` (lento, sobrecarregado pelo interpretador).
2. **NumPy:** Vetorizado e utilizando rotinas C/BLAS altamente otimizadas para CPU.


In [ ]:
# @title 🐍 Implementações CPU (Python Puro & NumPy)
import time
import numpy as np

# ── Versão 1: Python Puro ──────────────────────────────────
def soma_vetores_python(a, b):
    """Soma dois vetores elemento a elemento — laço Python puro."""
    resultado = [0.0] * len(a)
    for i in range(len(a)):
        resultado[i] = a[i] + b[i]
    return resultado

def multiplicacao_vetorial_python(a, b):
    """Produto escalar — redução sequencial em Python puro."""
    total = 0.0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

# ── Versão 2: CPU NumPy ───────────────────────────────────
def benchmark_numpy(N, repeticoes=50):
    a = np.ones(N, dtype=np.float32)
    b = np.arange(N, dtype=np.float32)

    # Warm-up
    _ = a + b
    _ = np.dot(a, b)

    # Benchmark soma
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        c = a + b
    t_soma = (time.perf_counter() - t0) / repeticoes

    # Benchmark dot
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        s = np.dot(a, b)
    t_dot = (time.perf_counter() - t0) / repeticoes

    return t_soma, t_dot

print("✅ Funções CPU (Python puro e NumPy) definidas com sucesso.")


## 3. Versão 3 — GPU: Kernels CUDA com Numba

Escrevemos kernels CUDA explícitos com Numba:
- **Kernel Soma:** Mapeia um thread para cada elemento do vetor (`cuda.grid(1)`).
- **Kernel Dot Product:** Realiza **redução paralela (Tree Reduction)** utilizando **Shared Memory** por bloco e atualização atômica no resultado final.


In [ ]:
# @title 🚀 Kernels CUDA com Numba (Soma Vetorial e Produto Escalar com Redução Paralela)
import math
import numba
import numba.cuda as cuda

# ── Kernel CUDA: Soma Vetorial ────────────────────────────
@cuda.jit
def kernel_soma(a, b, c):
    """Cada thread soma um elemento."""
    idx = cuda.grid(1)
    if idx < c.shape[0]:
        c[idx] = a[idx] + b[idx]

# ── Kernel CUDA: Produto Escalar (Tree Reduction em Shared Memory) ──
@cuda.jit
def kernel_dot_reducao(a, b, resultado_parcial):
    """
    Redução paralela em Shared Memory.
    Cada bloco soma parcialmente em árvore (Tree Reduction) e salva com atomicAdd.
    """
    shared = cuda.shared.array(shape=256, dtype=numba.float32)
    tx  = cuda.threadIdx.x
    idx = cuda.grid(1)

    val = 0.0
    if idx < a.shape[0]:
        val = a[idx] * b[idx]
    shared[tx] = val
    cuda.syncthreads()

    # Árvore de redução dentro do bloco
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tx < stride:
            shared[tx] += shared[tx + stride]
        cuda.syncthreads()
        stride //= 2

    # Thread 0 de cada bloco escreve o acumulado do bloco no global
    if tx == 0:
        cuda.atomic.add(resultado_parcial, 0, shared[0])

def benchmark_gpu_numba(N, repeticoes=50):
    a_h = np.ones(N, dtype=np.float32)
    b_h = np.arange(N, dtype=np.float32)

    a_d = cuda.to_device(a_h)
    b_d = cuda.to_device(b_h)
    c_d = cuda.device_array(N, dtype=np.float32)

    THREADS_POR_BLOCO = 256
    blocos = math.ceil(N / THREADS_POR_BLOCO)

    # Warm-up
    kernel_soma[blocos, THREADS_POR_BLOCO](a_d, b_d, c_d)
    res_d = cuda.to_device(np.zeros(1, dtype=np.float32))
    kernel_dot_reducao[blocos, THREADS_POR_BLOCO](a_d, b_d, res_d)
    cuda.synchronize()

    # Benchmark Soma
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        kernel_soma[blocos, THREADS_POR_BLOCO](a_d, b_d, c_d)
    cuda.synchronize()
    t_soma = (time.perf_counter() - t0) / repeticoes

    # Benchmark Dot Product
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        res_d = cuda.to_device(np.zeros(1, dtype=np.float32))
        kernel_dot_reducao[blocos, THREADS_POR_BLOCO](a_d, b_d, res_d)
    cuda.synchronize()
    t_dot = (time.perf_counter() - t0) / repeticoes

    return t_soma, t_dot

print("✅ Kernels Numba CUDA configurados com sucesso.")


## 4. Versão 4 — GPU: CuPy (NumPy Acelerado na GPU)

CuPy provê uma API idêntica ao NumPy rodando diretamente no CUDA da GPU sem a necessidade de escrever kernels manuais.


In [ ]:
# @title ⚡ Versão CuPy (NumPy na GPU)
import cupy as cp

def benchmark_cupy(N, repeticoes=50):
    a_cp = cp.ones(N, dtype=cp.float32)
    b_cp = cp.arange(N, dtype=cp.float32)

    # Warm-up
    _ = a_cp + b_cp
    _ = cp.dot(a_cp, b_cp)
    cp.cuda.runtime.deviceSynchronize()

    # Benchmark Soma
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        c_cp = a_cp + b_cp
    cp.cuda.runtime.deviceSynchronize()
    t_soma = (time.perf_counter() - t0) / repeticoes

    # Benchmark Dot Product
    t0 = time.perf_counter()
    for _ in range(repeticoes):
        s_cp = cp.dot(a_cp, b_cp)
    cp.cuda.runtime.deviceSynchronize()
    t_dot = (time.perf_counter() - t0) / repeticoes

    return t_soma, t_dot

print("✅ CuPy configurado com sucesso.")


## 5. Coleta Completa de Benchmark & Varredura de $N$

Varremos $N \in [10.000, 100.000, 1.000.000, 10.000.000, 100.000.000]$ efetuando no mínimo 50 repetições com warm-up prévio.


In [ ]:
# @title 📊 Executar Benchmark Completo das 4 Versões
import pandas as pd

Ns = [10_000, 100_000, 1_000_000, 10_000_000, 100_000_000]
repeticoes = 50

registros = []

print(f"{'N':>12} | {'Soma NumPy':>12} | {'Soma CUDA':>12} | {'Soma CuPy':>12} | {'Dot NumPy':>12} | {'Dot CUDA':>12} | {'Dot CuPy':>12}")
print("-" * 95)

for N in Ns:
    t_py_soma, t_py_dot = (np.nan, np.nan)
    if N <= 100_000:
        a_py = [1.0] * N
        b_py = [float(i) for i in range(N)]
        t0 = time.perf_counter()
        _ = soma_vetores_python(a_py, b_py)
        t_py_soma = time.perf_counter() - t0
        t0 = time.perf_counter()
        _ = multiplicacao_vetorial_python(a_py, b_py)
        t_py_dot = time.perf_counter() - t0

    ts_np, td_np = benchmark_numpy(N, repeticoes=repeticoes)
    ts_cuda, td_cuda = benchmark_gpu_numba(N, repeticoes=repeticoes)
    ts_cupy, td_cupy = benchmark_cupy(N, repeticoes=repeticoes)

    registros.append({
        "N": N,
        "Python_Soma_ms": t_py_soma * 1000 if not np.isnan(t_py_soma) else np.nan,
        "Python_Dot_ms": t_py_dot * 1000 if not np.isnan(t_py_dot) else np.nan,
        "NumPy_Soma_ms": ts_np * 1000,
        "NumPy_Dot_ms": td_np * 1000,
        "CUDA_Soma_ms": ts_cuda * 1000,
        "CUDA_Dot_ms": td_cuda * 1000,
        "CuPy_Soma_ms": ts_cupy * 1000,
        "CuPy_Dot_ms": td_cupy * 1000,
        "Speedup_Soma_CUDA": ts_np / ts_cuda,
        "Speedup_Soma_CuPy": ts_np / ts_cupy,
        "Speedup_Dot_CUDA": td_np / td_cuda,
        "Speedup_Dot_CuPy": td_np / td_cupy,
    })

    print(f"{N:>12,} | {ts_np*1000:>10.3f}ms | {ts_cuda*1000:>10.3f}ms | {ts_cupy*1000:>10.3f}ms | {td_np*1000:>10.3f}ms | {td_cuda*1000:>10.3f}ms | {td_cupy*1000:>10.3f}ms")

df_res = pd.DataFrame(registros)


## 6. Visualização Gráfica Interativa com Matplotlib

Geramos os gráficos comparativos de tempo absoluto e curva de speedup da GPU em relação ao NumPy.


In [ ]:
# @title 📈 Gráficos de Speedup e Tempo de Execução
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Benchmark de Desempenho: CPU (NumPy) vs GPU (CUDA Numba & CuPy)", fontsize=14, fontweight="bold")

# ── Gráfico 1: Tempos Absolutos de Soma Vetorial ──────────
ax1 = axes[0]
x_labels = [f"{n:,}" for n in df_res["N"]]
x = np.arange(len(x_labels))
largura = 0.25

ax1.bar(x - largura, df_res["NumPy_Soma_ms"], largura, label="NumPy (CPU)", color="#2563EB", alpha=0.85)
ax1.bar(x, df_res["CUDA_Soma_ms"], largura, label="CUDA Numba (GPU)", color="#10B981", alpha=0.85)
ax1.bar(x + largura, df_res["CuPy_Soma_ms"], largura, label="CuPy (GPU)", color="#8B5CF6", alpha=0.85)

ax1.set_xlabel("Tamanho do Vetor (N)")
ax1.set_ylabel("Tempo (ms) — escala log")
ax1.set_title("Tempo de Execução — Soma Vetorial")
ax1.set_xticks(x)
ax1.set_xticklabels(x_labels, rotation=30, ha="right")
ax1.set_yscale("log")
ax1.legend()
ax1.grid(axis="y", linestyle="--", alpha=0.4)

# ── Gráfico 2: Speedup (NumPy / GPU) ──────────────────────
ax2 = axes[1]
speedup_cuda = df_res["Speedup_Soma_CUDA"]
speedup_cupy = df_res["Speedup_Soma_CuPy"]

ax2.plot(x_labels, speedup_cuda, marker="o", linewidth=2, label="Speedup CUDA / NumPy", color="#10B981")
ax2.plot(x_labels, speedup_cupy, marker="s", linewidth=2, label="Speedup CuPy / NumPy", color="#8B5CF6")
ax2.axhline(y=1, color="gray", linestyle="--", linewidth=1.5, label="Sem ganho (1x)")

ax2.set_xlabel("Tamanho do Vetor (N)")
ax2.set_ylabel("Speedup (Tempo CPU / Tempo GPU)")
ax2.set_title("Curva de Speedup da GPU")
ax2.set_xticklabels(x_labels, rotation=30, ha="right")
ax2.legend()
ax2.grid(axis="both", linestyle="--", alpha=0.4)

for i, (sc, sp) in enumerate(zip(speedup_cuda, speedup_cupy)):
    ax2.annotate(f"{sc:.1f}x", (i, sc), textcoords="offset points", xytext=(0,6), ha='center', fontsize=8, fontweight='bold', color='#10B981')

plt.tight_layout()
plt.savefig("speedup_cpu_vs_gpu_aula13.png", dpi=150, bbox_inches="tight")
plt.show()
print("📸 Gráfico salvo como 'speedup_cpu_vs_gpu_aula13.png'")


## 7. Seção Extra: Requisito da Tarefa Final (`np.linalg.norm` vs `cp.linalg.norm`)

Conforme exigido na tarefa final do Bloco 2, avaliamos o cálculo de norma de vetor em CPU vs GPU.


In [ ]:
# @title 🧪 Benchmark Extra: `np.linalg.norm` vs `cp.linalg.norm`
N_extra = 10_000_000
a_np = np.ones(N_extra, dtype=np.float32)
a_cp = cp.ones(N_extra, dtype=cp.float32)

# Warm-up
_ = np.linalg.norm(a_np)
_ = cp.linalg.norm(a_cp)
cp.cuda.runtime.deviceSynchronize()

# Benchmark CPU
t0 = time.perf_counter()
for _ in range(30):
    norma_cpu = np.linalg.norm(a_np)
t_norm_cpu = (time.perf_counter() - t0) / 30

# Benchmark GPU
t0 = time.perf_counter()
for _ in range(30):
    norma_gpu = cp.linalg.norm(a_cp)
cp.cuda.runtime.deviceSynchronize()
t_norm_gpu = (time.perf_counter() - t0) / 30

print(f"N = {N_extra:,}")
print(f"Tempo `np.linalg.norm` (CPU): {t_norm_cpu * 1000:.3f} ms")
print(f"Tempo `cp.linalg.norm` (GPU): {t_norm_gpu * 1000:.3f} ms")
print(f"Speedup Norma Vetorial     : {t_norm_cpu / t_norm_gpu:.1f}x")


## 8. Mini-Relatório Técnico & Síntese Final do Bloco 2


In [ ]:
# @title 📑 Gerador de Mini-Relatório Técnico Automático
speedup_max = df_res["Speedup_Soma_CUDA"].max()
idx_max = df_res["Speedup_Soma_CUDA"].idxmax()
N_max = df_res.loc[idx_max, "N"]

idx_compensacao = df_res[df_res["Speedup_Soma_CUDA"] > 1.0].index.min()
N_limiar = df_res.loc[idx_compensacao, "N"] if pd.notna(idx_compensacao) else "N/A"

print(f"""
========================================================================
📊 MINI-RELATÓRIO TÉCNICO: DESEMPENHO PARALELO CPU VS GPU
========================================================================
1. RESUMO DOS RESULTADOS:
   - Speedup Máximo (NumPy ➔ CUDA): {speedup_max:.1f}x obtido em N = {N_max:,}.
   - Limiar de Eficiência da GPU: A GPU começou a superar o NumPy a partir de N = {N_limiar:,}.

2. ANÁLISE DE OVERHEAD DE MEMÓRIA:
   - Para tamanhos pequenos de vetor (N < 100K), a GPU apresenta desempenho inferior (speedup < 1x).
   - Isso ocorre devido ao custo fixo de lançamento de kernels (launch overhead) e ao transporte
     de dados via barramento PCIe (Host ➔ Device).

3. COMPARATIVO DAS ABORDAGENS:
   - Python Puro: Inviável para workloads de grande porte (>100K iter/s gargalo no interpretador).
   - NumPy (CPU): Excelente para protótipos e vetores pequenos/médios graças às rotinas BLAS.
   - CuPy (GPU): API idêntica ao NumPy com speedup imediato sem verbosidade de código.
   - Numba CUDA (GPU): Controle total sobre a hierarquia de threads e memória compartilhada.

4. RESPOSTA À TAREFA FINAL:
   No hardware testado, a GPU passa a compensar a partir de N ≈ {N_limiar:,}, atingindo
   sua maior eficiência em N ≥ 10M, onde a densidade computacional oculta a latência de transferência.
========================================================================
""")


---
## 📝 9. Lista de 20 Exercícios Práticos e Teóricos (Consolidação do Bloco 2)

Resolva os exercícios abaixo diretamente no notebook ou responda às questões concituais em formato de texto para fixação.

### 🔹 Parte 1: Exercícios Conceituais e Teóricos (1 a 10)
1. **Qual a principal diferença entre o modelo de processamento SIMD (CPU) e SIMT (GPU)?**
2. **Por que para vetores pequenos ($N < 100.000$) a CPU com NumPy pode ser mais rápida que a GPU?**
3. **Explique a função do barramento PCIe na transferência de dados entre RAM (Host) e VRAM (Device).**
4. **O que é e qual o papel do *Warm-up* antes de realizar medições de tempo e benchmark de kernels CUDA?**
5. **Diferencie a Memória Global (VRAM) da Memória Compartilhada (Shared Memory) em uma GPU NVIDIA.**
6. **O que é um Warp em arquitetura CUDA e qual o seu tamanho em número de threads?**
7. **Explique a técnica de *Tree Reduction* (redução em árvore) utilizada no produto escalar.**
8. **Por que usamos `cuda.atomic.add` ao consolidar os resultados parciais dos blocos em uma redução?**
9. **Qual a vantagem técnica de utilizar CuPy em relação ao desenvolvimento manual de kernels Numba CUDA?**
10. **Em que situações práticas de engenharia de IA um desenvolvedor PRECISE escrever um kernel customizado em Numba/CUDA em vez de usar CuPy ou PyTorch?**

---
### 🔹 Parte 2: Exercícios Práticos e Mão na Massa (11 a 20)


In [ ]:
# @title ✏️ Exercício 11: Modificar Threads por Bloco
# Modifique o parâmetro THREADS_POR_BLOCO de 256 para 128 e depois para 512 na função `benchmark_gpu_numba`.
# Qual valor apresentou a melhor performance para N = 10_000_000?

N_ex11 = 10_000_000
# ESCREVA OU EXECUTE O CÓDIGO AQUI


In [ ]:
# @title ✏️ Exercício 12: Subtração Vetorial CUDA Numba
# Escreva um kernel CUDA `@cuda.jit` chamado `kernel_subtracao(a, b, c)` que calcula c[i] = a[i] - b[i].
# Teste com N = 1_000_000 e verifique o resultado comparando com NumPy.

# ESCREVA O KERNEL E O TESTE AQUI


In [ ]:
# @title ✏️ Exercício 13: Multiplicação Elemento a Elemento em CuPy
# Crie dois vetores CuPy com 5.000.000 de elementos float32 e meça o tempo para multiplicar a * b.
# Compare com o tempo equivalente em NumPy CPU.

# ESCREVA SEU CÓDIGO AQUI


In [ ]:
# @title ✏️ Exercício 14: Cálculo de Média Vetorial em GPU
# Implemente o cálculo da média dos elementos de um vetor de tamanho N=10M usando CuPy (`cp.mean`).
# Calcule o speedup em relação ao `np.mean` da CPU.

# ESCREVA SEU CÓDIGO AQUI


In [ ]:
# @title ✏️ Exercício 15: Aplicação de Função Ativação ReLU em CUDA Numba
# Escreva um kernel CUDA `@cuda.jit` que aplica a função ReLU: c[i] = max(0.0, a[i]).
# Teste com um vetor contendo valores positivos e negativos.

# ESCREVA SEU CÓDIGO AQUI


In [ ]:
# @title ✏️ Exercício 16: Medição da Transferência de Memória PCIe (Host -> Device -> Host)
# Utilize `time.perf_counter()` para medir isoladamente:
# 1. O tempo de transferência de um vetor NumPy 100MB da CPU para a GPU (`cuda.to_device`).
# 2. O tempo de execução do kernel de soma.
# 3. O tempo de retorno da GPU para a CPU (`c_d.copy_to_host()`).
# Qual etapa levou mais tempo?

# ESCREVA SEU CÓDIGO AQUI


In [ ]:
# @title ✏️ Exercício 17: Comparação de Precisão float32 vs float64 em CuPy
# Crie vetores CuPy de tamanho N = 10_000_000 em `cp.float32` e `cp.float64`.
# Meça o tempo da soma vetorial em ambos. A GPU processa float32 mais rápido que float64? Quanto mais rápido?

# ESCREVA SEU CÓDIGO AQUI


In [ ]:
# @title ✏️ Exercício 18: Normalização Min-Max Vetorial em CuPy
# Implemente a normalização Min-Max $v_{norm} = (v - min) / (max - min)$ para um vetor CuPy N=5M.
# Meça o tempo total e compare com a implementação em NumPy.

# ESCREVA SEU CÓDIGO AQUI


In [ ]:
# @title ✏️ Exercício 19: Operação Combinada $c[i] = a[i] \cdot x + b[i]$ (SAXPY/DAXPY)
# Escreva um kernel CUDA `@cuda.jit` que implementa a operação AXPY: c[i] = alpha * a[i] + b[i].
# Teste com N = 10_000_000 e alpha = 2.5.

# ESCREVA SEU CÓDIGO AQUI


In [ ]:
# @title ✏️ Exercício 20: Desafio Final — Validação do Limiar de Speedup em Gráfico Customizado
# Execute o benchmark para N = [1.000, 5.000, 20.000, 50.000, 200.000, 500.000].
# Plote um gráfico focado EXATAMENTE no ponto em que o Speedup cruza a linha de 1.0x (ponto de equilíbrio).

# ESCREVA SEU CÓDIGO E PLOTE O GRÁFICO AQUI
